In [2]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# 1. Connect to database
db_path = Path(
    "D:/NordicFlow Project - Full ERP project/"
    "Project-05-Python-ERP-Analytics/data/NordicFlow_ERP.db"
)

conn = sqlite3.connect(db_path)

# 2. Load ERP tables
material = pd.read_sql_query(
    "SELECT * FROM material_master",
    conn
)

supplier = pd.read_sql_query(
    "SELECT * FROM supplier_master",
    conn
)

plant = pd.read_sql_query(
    "SELECT * FROM plant_master",
    conn
)

bom = pd.read_sql_query(
    "SELECT * FROM bom",
    conn
)

inventory = pd.read_sql_query(
    "SELECT * FROM inventory_snapshot",
    conn
)

purchase_orders = pd.read_sql_query(
    "SELECT * FROM purchase_order_lines",
    conn
)

production = pd.read_sql_query(
    "SELECT * FROM production_orders",
    conn
)

# 3. Convert currency fields to numeric
inventory["Unit_Cost_EUR"] = pd.to_numeric(
    inventory["Unit_Cost_EUR"]
    .astype(str)
    .str.replace(r"[^\d.-]", "", regex=True),
    errors="coerce"
)

inventory["Inventory_Value_EUR"] = pd.to_numeric(
    inventory["Inventory_Value_EUR"]
    .astype(str)
    .str.replace(r"[^\d.-]", "", regex=True),
    errors="coerce"
)

purchase_orders["Unit_Price_EUR"] = pd.to_numeric(
    purchase_orders["Unit_Price_EUR"]
    .astype(str)
    .str.replace(r"[^\d.-]", "", regex=True),
    errors="coerce"
)

material["Standard_Cost_EUR"] = pd.to_numeric(
    material["Standard_Cost_EUR"]
    .astype(str)
    .str.replace(r"[^\d.-]", "", regex=True),
    errors="coerce"
)

# 4. Check material table
material.head()

# 5. Verify record counts
tables = {
    "Material Master": material,
    "Supplier Master": supplier,
    "Plant Master": plant,
    "BOM": bom,
    "Inventory Snapshot": inventory,
    "Purchase Orders": purchase_orders,
    "Production Orders": production
}

for name, df in tables.items():
    print(name, len(df))

# 6. Check missing values
for name, df in tables.items():
    print("\n", name)
    print(df.isnull().sum())

# 7. Duplicate checks
print(material["Material_ID"].duplicated().sum())
print(supplier["Supplier_ID"].duplicated().sum())
print(production["Production_Order_ID"].duplicated().sum())

# 8. Confirm numeric conversion
print(inventory["Inventory_Value_EUR"].head())

# 9. Calculate latest inventory value
latest_date = inventory["Snapshot_Date"].max()

latest_inventory = inventory[
    inventory["Snapshot_Date"] == latest_date
]

total_inventory_value = latest_inventory[
    "Inventory_Value_EUR"
].sum()

print("Latest inventory date:", latest_date)
print("Total inventory value:", total_inventory_value)


# Display all columns in Purchase Orders
# print(purchase_orders.columns.tolist())

# Convert purchase order dates using day/month/year format
purchase_orders["Required_Delivery_Date"] = pd.to_datetime(
    purchase_orders["Required_Delivery_Date"],
    dayfirst=True
)

purchase_orders["Actual_Receipt_Date"] = pd.to_datetime(
    purchase_orders["Actual_Receipt_Date"],
    dayfirst=True
)

# Create derived on-time delivery flag
purchase_orders["On_Time_Flag_Calc"] = (
    purchase_orders["Actual_Receipt_Date"]
    <= purchase_orders["Required_Delivery_Date"]
)


# Convert TRUE/FALSE text into 1/0
production["Material_Shortage_Flag_Calc"] = (
    production["Material_Shortage_Flag"]
    .astype(str)
    .str.strip()
    .str.upper()
    .map({
        "TRUE": 1,
        "FALSE": 0,
        "1": 1,
        "0": 0
    })
)


# Supplier On-Time Delivery %
supplier_otd = (
    purchase_orders["On_Time_Flag_Calc"]
    .mean()
    * 100
)

# Production On-Time Completion %
production_otd = (
    (production["Completion_Delay_Days"] <= 0)
    .mean()
    * 100
)

# Number of production orders affected by material shortages
shortage_orders = (
    production["Material_Shortage_Flag_Calc"]
    .sum()
)

# Build management KPI table
kpi_summary = pd.DataFrame({
    "KPI": [
        "Latest Inventory Value EUR",
        "Supplier On-Time Delivery %",
        "Production On-Time Completion %",
        "Shortage-Affected Orders"
    ],
    "Value": [
        round(total_inventory_value, 2),
        round(supplier_otd, 1),
        round(production_otd, 1),
        int(shortage_orders)
    ]
})

# Display all KPIs
print(kpi_summary)

print(f"Supplier OTD: {supplier_otd:.1f}%")

output_path = Path("D:/NordicFlow Project - Full ERP project/Project-05-Python-ERP-Analytics/outputs")

output_path.mkdir(exist_ok=True)

kpi_summary.to_csv(
    output_path / "kpi_summary.csv",
    index=False
)

# -------------------------------------------------
# STEP 8 — Prepare latest inventory snapshot
# -------------------------------------------------

# Identify the most recent inventory snapshot date
latest_date = inventory["Snapshot_Date"].max()

# Keep only the latest snapshot
latest_inventory = inventory[
    inventory["Snapshot_Date"] == latest_date
].copy()

print("Latest inventory snapshot:", latest_date)
print("Rows in latest snapshot:", len(latest_inventory))

# -------------------------------------------------
# STEP 9 — Aggregate unrestricted inventory
# -------------------------------------------------

# Keep only stock that is immediately usable
unrestricted_inventory = (
    latest_inventory[
        latest_inventory["Stock_Status"] == "Unrestricted"
    ]
    .groupby(
        ["Material_ID", "Plant_ID"],
        as_index=False
    )
    .agg(
        Unrestricted_Qty=("Quantity", "sum"),
        Inventory_Value_EUR=("Inventory_Value_EUR", "sum")
    )
)

unrestricted_inventory.head()

# -------------------------------------------------
# STEP 10 — Join inventory with material planning data
# -------------------------------------------------

inventory_risk = unrestricted_inventory.merge(
    material[
        [
            "Material_ID",
            "Material_Name",
            "Safety_Stock_Qty",
            "Reorder_Point_Qty",
            "ABC_Class",
            "XYZ_Class",
            "Criticality",
            "Preferred_Supplier_ID"
        ]
    ],
    on="Material_ID",
    how="left"
)

inventory_risk.head()

# -------------------------------------------------
# STEP 11 — Detect materials below safety stock
# -------------------------------------------------

inventory_risk["Below_Safety_Stock"] = (
    inventory_risk["Unrestricted_Qty"]
    < inventory_risk["Safety_Stock_Qty"]
)

inventory_risk["Safety_Stock_Gap"] = (
    inventory_risk["Safety_Stock_Qty"]
    - inventory_risk["Unrestricted_Qty"]
)

safety_stock_shortages = inventory_risk[
    inventory_risk["Below_Safety_Stock"]
].copy()

safety_stock_shortages[
    [
        "Plant_ID",
        "Material_ID",
        "Material_Name",
        "Criticality",
        "Unrestricted_Qty",
        "Safety_Stock_Qty",
        "Safety_Stock_Gap"
    ]
].sort_values(
    ["Criticality", "Safety_Stock_Gap"],
    ascending=[True, False]
)

# -------------------------------------------------
# STEP 12 — Detect materials below reorder point
# -------------------------------------------------

inventory_risk["Below_Reorder_Point"] = (
    inventory_risk["Unrestricted_Qty"]
    < inventory_risk["Reorder_Point_Qty"]
)

inventory_risk["Reorder_Gap"] = (
    inventory_risk["Reorder_Point_Qty"]
    - inventory_risk["Unrestricted_Qty"]
)

reorder_risk = inventory_risk[
    inventory_risk["Below_Reorder_Point"]
].copy()

reorder_risk.head(10)

# -------------------------------------------------
# STEP 13 — Detect potential C-class overstock
# -------------------------------------------------

c_class_overstock = inventory_risk[
    (inventory_risk["ABC_Class"] == "C")
    &
    (
        inventory_risk["Unrestricted_Qty"]
        > inventory_risk["Reorder_Point_Qty"]
    )
].copy()

c_class_overstock["Excess_Qty"] = (
    c_class_overstock["Unrestricted_Qty"]
    - c_class_overstock["Reorder_Point_Qty"]
)

c_class_overstock[
    [
        "Plant_ID",
        "Material_ID",
        "Material_Name",
        "Unrestricted_Qty",
        "Reorder_Point_Qty",
        "Excess_Qty",
        "Inventory_Value_EUR"
    ]
].sort_values(
    "Inventory_Value_EUR",
    ascending=False
)

# -------------------------------------------------
# STEP 14 — Identify interplant transfer opportunities
# -------------------------------------------------

transfer_candidates = inventory_risk.merge(
    inventory_risk[
        [
            "Material_ID",
            "Plant_ID",
            "Unrestricted_Qty",
            "Safety_Stock_Qty"
        ]
    ],
    on="Material_ID",
    suffixes=("_Shortage", "_Source")
)

# Keep only different plants
transfer_candidates = transfer_candidates[
    transfer_candidates["Plant_ID_Shortage"]
    != transfer_candidates["Plant_ID_Source"]
]

# Shortage plant must be below safety stock
# Source plant must be above safety stock
transfer_candidates = transfer_candidates[
    (
        transfer_candidates["Unrestricted_Qty_Shortage"]
        < transfer_candidates["Safety_Stock_Qty_Shortage"]
    )
    &
    (
        transfer_candidates["Unrestricted_Qty_Source"]
        > transfer_candidates["Safety_Stock_Qty_Source"]
    )
].copy()

# Calculate indicative transferable quantity
transfer_candidates["Indicative_Transfer_Qty"] = transfer_candidates[
    [
        "Safety_Stock_Gap",
        "Unrestricted_Qty_Source"
    ]
].min(axis=1)

transfer_candidates[
    [
        "Material_ID",
        "Material_Name",
        "Plant_ID_Shortage",
        "Unrestricted_Qty_Shortage",
        "Plant_ID_Source",
        "Unrestricted_Qty_Source"
    ]
]

# -------------------------------------------------
# STEP 15 — Calculate realistic inter-plant transfer quantity
# -------------------------------------------------

# Calculate surplus available at the source plant
transfer_candidates["Source_Surplus_Qty"] = (
    transfer_candidates["Unrestricted_Qty_Source"]
    - transfer_candidates["Safety_Stock_Qty_Source"]
)

# Calculate shortage at the receiving plant
transfer_candidates["Shortage_Qty"] = (
    transfer_candidates["Safety_Stock_Qty_Shortage"]
    - transfer_candidates["Unrestricted_Qty_Shortage"]
)

# Recommended transfer = smaller of shortage or source surplus
transfer_candidates["Indicative_Transfer_Qty"] = (
    transfer_candidates[
        ["Shortage_Qty", "Source_Surplus_Qty"]
    ]
    .min(axis=1)
)

# Keep only positive transfer opportunities
interplant_transfer = transfer_candidates[
    transfer_candidates["Indicative_Transfer_Qty"] > 0
].copy()

# Show the final transfer report
interplant_transfer[
    [
        "Material_ID",
        "Material_Name",
        "Plant_ID_Shortage",
        "Unrestricted_Qty_Shortage",
        "Safety_Stock_Qty_Shortage",
        "Plant_ID_Source",
        "Unrestricted_Qty_Source",
        "Safety_Stock_Qty_Source",
        "Indicative_Transfer_Qty"
    ]
]

# -------------------------------------------------
# STEP 16 — Export inventory analysis outputs
# -------------------------------------------------

inventory_risk.to_csv(
    output_path / "inventory_risk.csv",
    index=False
)

safety_stock_shortages.to_csv(
    output_path / "safety_stock_shortages.csv",
    index=False
)

reorder_risk.to_csv(
    output_path / "reorder_point_risk.csv",
    index=False
)

c_class_overstock.to_csv(
    output_path / "c_class_overstock.csv",
    index=False
)

interplant_transfer.to_csv(
    output_path / "interplant_transfer_opportunities.csv",
    index=False
)

print("Inventory analysis files exported successfully.")

Material Master 20
Supplier Master 8
Plant Master 3
BOM 33
Inventory Snapshot 271
Purchase Orders 24
Production Orders 15

 Material Master
Material_ID               0
Material_Name             0
Material_Type             0
Material_Group            0
Base_Unit                 0
Standard_Cost_EUR         0
Planned_Lead_Time_Days    0
Safety_Stock_Qty          0
Reorder_Point_Qty         0
ABC_Class                 0
XYZ_Class                 0
Criticality               0
Preferred_Supplier_ID     5
Active_Status             0
dtype: int64

 Supplier Master
Supplier_ID              0
Supplier_Name            0
Supplier_Country         0
Supplier_Category        0
Payment_Terms_Days       0
Risk_Level               0
Preferred_Status         0
Active_Status            0
Target_Lead_Time_Days    0
Target_On_Time_Rate      0
dtype: int64

 Plant Master
Plant_ID         0
Plant_Name       0
City             0
Plant_Type       0
Country          0
Active_Status    0
dtype: int64

 BOM
BOM_ID